In [13]:
import h5py
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib import colors
from os.path import join, realpath, basename
import json

#### Tools

In [14]:
"""
common_utils.py
----------------
Shared utilities and lightweight container(s) for HDF5 readers.
"""
from dataclasses import dataclass
from typing import Optional, Any, Dict
import numpy as np
import h5py


@dataclass
class LC:
    time: np.ndarray
    values: np.ndarray
    variance: Optional[np.ndarray] = None
    meta: Dict[str, Any] = None
    
def walk(h, prefix=""):
    for name, obj in h.items():
        path = f"{prefix}/{name}" if prefix else name
        if isinstance(obj, h5py.Group):
            print(f"[Group]   {path}  (members: {len(obj)})")
            walk(obj, path)
        else:  # Dataset
            print(f"[Dataset] {path}  shape={obj.shape} dtype={obj.dtype} attrs={list(obj.attrs.keys())}")

In [15]:
"""
psf_reader.py
-------------
Readers for PSF products using canonical time in /products/psf/time/Q{q}/g{g}_time.
"""
import h5py
import numpy as np
from typing import Dict

def load_psf_target(h5: h5py.File, mask: str = "target") -> Dict[int, Dict[int, LC]]:
    """
    Load PSF target flux per quarter and per group.
    Returns: {quarter: {group: LC(time, flux, flux_variance)}}
    Group 0 = total quarter‐level (flux/variance) if present.
    Groups 1..N = individual groups with their time, flux, variance.
    """
    root = f"/products/psf/{mask}"
    out: Dict[int, Dict[int, LC]] = {}
    if root not in h5:
        return out

    g_time = load_time(h5, product="psf")  # {quarter: {group: time_array}}

    branch = h5[root]
    for qname, qgrp in branch.items():
        if not qname.startswith("Q"):
            continue
        q = int(qname[1:])
        out[q] = {}

        # ——— Handle individual groups first so we know available groups & times
        group_keys = []
        for gname, ggrp in qgrp.items():
            if not gname.startswith("group"):
                continue
            try:
                g = int(gname[5:])
            except ValueError:
                continue
            group_keys.append(g)
            # Retrieve time for this group
            time_arr = None
            if q in g_time and g in g_time[q]:
                time_arr = g_time[q][g]
            if time_arr is None:
                # skip if no time
                print(f" Quarter {q} group {g}: missing time array, skipping group.")
                continue
            # Retrieve flux & variance
            flux_arr = ggrp["flux"][()] if "flux" in ggrp else np.array([])
            var_arr  = ggrp["flux_variance"][()] if "flux_variance" in ggrp else None
            out[q][g] = LC(time=time_arr,
                           values=flux_arr,
                           variance=var_arr,
                           meta={"product":"psf","mask":mask,"quarter":q,"group":g})

        # ——— Now handle total flux / variance (group 0) if present
        if "flux" in qgrp:
            # only add total if flux dataset present at quarter level
            flux_arr = qgrp["flux"][()]
            var_arr  = qgrp["flux_variance"][()] if "flux_variance" in qgrp else None

            # For time of total: if you prefer to pick one of the groups to align with,
            # pick first available group time if exists, else none.
            time_arr = None
            if group_keys:
                # choose smallest group number
                g0 = min(group_keys)
                if q in g_time and g0 in g_time[q]:
                    time_arr = g_time[q][g0]

            out[q][0] = LC(time=time_arr,
                           values=flux_arr,
                           variance=var_arr,
                           meta={"product":"psf","mask":mask,"quarter":q,"group":0})
        else:
            # optional: warn about missing total
            print(f"Quarter {q}: no quarter‐level ‘flux’ dataset, so total (group 0) not built.")

    return out


def load_psf_contaminant(h5: h5py.File, cid: int, mask: str = "psf") -> Dict[int, Dict[int, LC]]:
    """
    Load PSF contaminant flux per quarter and per group for a given contaminant id.
    Returns: {quarter: {group: LC(time, flux, flux_variance)}}
    Group 0 = total quarter‐level (flux/variance) if present.
    Groups 1..N = individual groups with their time, flux, variance.
    """
    root = f"/products/psf/contaminants/{int(cid)}"

    out: Dict[int, Dict[int, LC]] = {}
    if root not in h5:
        print(f"Warning : contaminant {int(cid)} does not exist - please check")
        return out

    g_time = load_time(h5, product="psf")  # {quarter: {group: time_array}}

    branch = h5[root]
    for qname, qgrp in branch.items():
        if not qname.startswith("Q"):
            continue
        q = int(qname[1:])
        out[q] = {}

        # ——— Handle individual groups first so we know available groups & times
        group_keys = []
        for gname, ggrp in qgrp.items():
            if not gname.startswith("group"):
                continue
            try:
                g = int(gname[5:])
            except ValueError:
                continue
            group_keys.append(g)
            # Retrieve time for this group
            time_arr = None
            if q in g_time and g in g_time[q]:
                time_arr = g_time[q][g]
            if time_arr is None:
                # skip if no time
                print(f" Quarter {q} group {g}: missing time array, skipping group.")
                continue
            # Retrieve flux & variance
            flux_arr = ggrp["flux"][()] if "flux" in ggrp else np.array([])
            var_arr  = ggrp["flux_variance"][()] if "flux_variance" in ggrp else None
            out[q][g] = LC(time=time_arr,
                           values=flux_arr,
                           variance=var_arr,
                           meta={"product":"psf","mask":mask,"quarter":q,"group":g})

        # ——— Now handle total flux / variance (group 0) if present
        if "flux" in qgrp:
            # only add total if flux dataset present at quarter level
            flux_arr = qgrp["flux"][()]
            var_arr  = qgrp["flux_variance"][()] if "flux_variance" in qgrp else None

            # For time of total: if you prefer to pick one of the groups to align with,
            # pick first available group time if exists, else none.
            time_arr = None
            if group_keys:
                # choose smallest group number
                g0 = min(group_keys)
                if q in g_time and g0 in g_time[q]:
                    time_arr = g_time[q][g0]

            out[q][0] = LC(time=time_arr,
                           values=flux_arr,
                           variance=var_arr,
                           meta={"product":"psf","mask":mask,"quarter":q,"group":0,"cid":int(cid)})
        else:
            # optional: warn about missing total
            print(f"Quarter {q}: no quarter‐level ‘flux’ dataset, so total (group 0) not built.")

    return out


    
    
def load_psf_contaminant_old(h5: h5py.File, cid: int, mask: str = "psf") -> Dict[int, Dict[int, LC]]:
    """
    Load PSF contaminant flux per quarter and per group for a given contaminant id.
    Returns: {quarter: {group: LC}}
    """
    root = f"/products/psf/contaminants/{int(cid)}"
    out: Dict[int, Dict[int, LC]] = {}
    if root not in h5:
        return out

    for qname, qgrp in h5[root].items():
        if not qname.startswith("Q"):
            continue
        q = int(qname[1:])
        out[q] = {}
        for gname, ggrp in qgrp.items():
            if not gname.startswith("group"):
                continue
            g = int(gname[5:])
            try:
                time = get_time_for(h5, "psf", q, g)
            except KeyError:
                continue
            var = ggrp.get("flux_variance")
            out[q][g] = LC(time=time,
                           values=ggrp.get("flux", np.array([]))[()],
                           variance=(var[()] if var is not None else None),
                           meta={"product":"psf","mask":mask,"quarter":q,"group":g,"cid":int(cid)})
    return out

In [16]:
"""
aperture_reader.py
------------------
Readers for aperture products using canonical time in /products/aperture/time/Q{q}/g{g}_time.
"""
import h5py
import numpy as np
from typing import Dict

def load_aperture_target(h5: h5py.File, mask: str = "nominal") -> Dict[int, Dict[int, LC]]:
    """
    Load aperture target flux per quarter and per group.
    Returns: {quarter: {group: LC(time, flux, flux_variance)}}
    """
    root = f"/products/aperture/{mask}"
    out: Dict[int, Dict[int, LC]] = {}
    if root not in h5:
        return out

    g_time = load_time(h5, product="aperture")

    branch = h5[root]
    for qname, qgrp in branch.items():
        if not qname.startswith("Q"):
            continue
        q = int(qname[1:])
        out[q] = {}
        # quarter-level totals (if present)
        if "flux" in qgrp:
            # choose canonical time: first group time in this quarter (or None if absent)
            t_q = next(iter(g_time.get(q, {}).values()), None)
            if t_q is not None:
                var = qgrp.get("flux_variance")
                out[q][0] = LC(time=t_q,
                               values=qgrp["flux"][()],
                               variance=(var[()] if var is not None else None),
                               meta={"product":"aperture","mask":mask,"quarter":q,"group":0})
        # per-group
        for gname, ggrp in qgrp.items():
            if not gname.startswith("group"):
                continue
            g = int(gname[5:])
            try:
                time = get_time_for(h5, "aperture", q, g)
            except KeyError:
                continue
            var = ggrp.get("flux_variance")
            if "flux" in ggrp:
                vals = ggrp["flux"][()]
            else:
                vals = np.array([])
            out[q][g] = LC(time=time,
                           values=vals,
                           variance=(var[()] if var is not None else None),
                           meta={"product":"aperture","mask":mask,"quarter":q,"group":g})
    return out

def load_aperture_cob(h5: h5py.File, mask: str = "nominal") -> Dict[int, Dict[int, dict]]:
    """
    Load aperture COB per quarter and per group.
    Returns: {quarter: {group: {"cob_x": ..., "cob_y": ..., "cob_x_variance": ..., "cob_y_variance": ...}}}
    """
    root = f"/products/aperture/{mask}/cob"
    out: Dict[int, Dict[int, dict]] = {}
    if root not in h5:
        return out

    for qname, qgrp in h5[root].items():
        if not qname.startswith("Q"):
            continue
        q = int(qname[1:])
        out[q] = {}
        # quarter-level totals (if present)
        totals = {}
        for k in ("cob_x","cob_x_variance","cob_y","cob_y_variance"):
            if k in qgrp:
                totals[k] = qgrp[k][()]
        if totals:
            out[q][0] = totals

        # per-group
        for gname, ggrp in qgrp.items():
            if not gname.startswith("group"):
                continue
            g = int(gname[5:])
            payload = {}
            for k in ("cob_x","cob_x_variance","cob_y","cob_y_variance"):
                if k in ggrp:
                    payload[k] = ggrp[k][()]
            if payload:
                out[q][g] = payload
    return out

#### Code

In [17]:
from time_utils import load_time, get_time_for, list_available_times
#from psf_reader import load_psf_target, load_psf_contaminant, load_psf_contaminant_old
#from aperture_reader import load_aperture_target, load_aperture_cob
from all_attributes import (
    read_target, 
    read_target_catalog,
    read_obs_conditions_summary_list, 
    print_obs_quarters, 
    list_contaminants_prop
)
from aperture_metrics import (
    list_aperture_contaminants,       # could be none or 2, 3, 4, ...
    load_aperture_metrics_table,      # global metrics_table 
    load_aperture_contaminant_qgc,    # {cid: qgc_table} - q= quarter, g = group, c= camera
    load_aperture_contaminant_metrics # dict with global, per-cid, and concatenated qgc
)
from psf_metrics import (
    list_psf_metrics_contaminants,  
    load_psf_metrics_table,         
    load_psf_contaminant_qgc,       
    load_psf_metrics,               
)
from df_utils import (
    lc_dict_to_df,
    lc_dict_to_quarter,
    concat_quarter,
    cob_dict_to_quarter
)
from read_allparams import(read_all,all_params)

from common_utils import walk

In [18]:
sim_start = "00000"
sim_end = "00420"
root_sim = f"{sim_start}-{sim_end}" 
path_local = f"/home/mdeleuil/plato/SimusOut"
#path_local = f"/plato/MakeSimus/simus_00251-00310/"
#simus_{root_sim}"
path="/net/GSP/nas12c/plato/FichiersParams/"
all,conts = read_all(path,sim_start, sim_end)

AllParameters file: /net/GSP/nas12c/plato/FichiersParams/sim00000_00420_AllParameters.ftr
AllParameters_conts file: /net/GSP/nas12c/plato/FichiersParams/sim00000_00420_AllParameters_Conts.ftr


In [19]:
def get_group_names(file_path):
    groups_founds = []
    for i in range (1,5):
        groups_found = []
        base_path = f"products/aperture/nominal/Q{i}"
        with h5py.File(file_path, "r") as h5:
            # Accès direct au dossier Q3
            if base_path in h5:
                parent = h5[base_path]
                # On liste tout ce qui est un Groupe et qui contient "group" dans le nom
                for name in parent:
                # On vérifie si c'est un groupe et s'il contient "group"
                    if isinstance(parent[name], h5py.Group) and "group" in name:
                    # On remplace "group" par rien pour garder le numéro
                        number = name.replace("group", "")
                        groups_found.append(number)
            groups_founds.append(groups_found)
    return groups_founds

In [20]:
def get_quarter_sky(file_path):
    results = []
    
    with h5py.File(file_path, "r") as h5:
        for i in range(1, 5):
            # On teste les deux types de dossiers
            for mode in ["nominal", "extended"]:
                path = f"products/aperture/{mode}/sky/Q{i}"
                
                if path in h5:
                    parent = h5[path]
                    # On parcourt les sous-groupes (group1, group3, etc.)
                    for name in parent:
                        group_obj = parent[name]
                        
                        if isinstance(group_obj, h5py.Group):
                            # On vérifie si les datasets de déplacement sont à l'intérieur
                            # Adapté selon ta hiérarchie : products/aperture/.../Q1/group3/camera1/gcrs_dlat
                            # Ici on vérifie simplement si le sous-groupe contient des données
                            if "camera1" in group_obj: # Ou un autre critère de présence
                                results.append(f"Q{i}_{mode}_{name.replace('group', 'g')}")
                                
    return results

In [21]:
def ang_dist (dlt1,dlt2,alp1,alp2):
    ra1 = alp1 * np.pi / 180
    ra2 = alp2 * np.pi / 180
    dec1 = dlt1 * np.pi / 180
    dec2 = dlt2 * np.pi / 180
    return (np.arccos(np.cos(dec1) * np.cos(dec2) * (np.cos(ra1) * np.cos(ra2) + np.sin(ra1) * np.sin(ra2)) + np.sin(dec1) * np.sin(dec2)) * 3600 * 180 / np.pi)

In [22]:
num_sim=[60,63,64,67,70,80,84,85,90,91,
    99,104,108,110,112,118,120,122,123,125,130,132,136,142,149,152,155,157,159,160,163,170,171,176,
    181,186,187,190,191,201,205,208,209,212,213,214,216,220,221,227,228,230,236,240,241,245,246,247]
num_cont=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 
          36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]

In [3]:
num_sim.index(221)

48

In [4]:
num_cont[48]

54

In [60]:
nb=170

In [12]:
# Takes into account the contaminants even if the cont_type is false
nb=349 #simulation number 
if nb<100:
    simu=f"000{nb}"
elif nb>=100:
    simu=f"00{nb}"
file_name=f"sim{simu}"
file_simus = f"{path_local}/{file_name}.hdf5"
a=0
try:
    nc=num_cont[num_sim.index(nb)]
    print(f"Trouvé à l'indice {nc}")
except ValueError:
    # On entre ici uniquement si .index() a échoué
    print("Pas de contaminant!")
    a=1
# 1. Définition des chemins (Les feather sont déjà en panda.dataframe)
file_feather_all = '/net/GSP/nas12c/plato/FichiersParams/sim00000_00420_AllParameters.ftr'
file_feather_conts = '/net/GSP/nas12c/plato/FichiersParams/sim00000_00420_AllParameters_Conts.ftr'
file_simus_hdf5 = file_simus

# --- CHARGEMENT DES DONNÉES ---
df_targets = pd.read_feather(file_feather_all)
df_conts = pd.read_feather(file_feather_conts)

#---- Mise à -1 pour voir où es-ce qu'il n'y a pas de donner----
ncams,depth,depth_sec,dist,ncont,mag_prim=-1,-1,-1,-1,-1,-1
mag_sec=[-1]

# --- CALCULS ET STATISTIQUES ---
ncams = df_targets['ncams'].iloc[nb]
#print (ncam)
ncont = df_targets['ncon'].iloc[nb]
#print (ncon)
mag_prim = df_targets['mag'].iloc[nb]
#print (mag)
if a==0:
    with h5py.File(file_simus, "r") as h5:
        conta = list_contaminants_prop(h5)
        print(conta)
    mag_sec=[]
    for i in range (len(conta)):
        mag_sec.append(conta[i][-4])
    print(mag_sec)
    dist = ang_dist(df_conts['dec'].iloc[nc],df_targets['dec'].iloc[nb],df_conts['ra'].iloc[nc],df_targets['ra'].iloc[nb])
            
if df_targets['signal_type'].iloc[nb]=="planet":
    depth = df_targets['depth_ldc'].iloc[nb]
    depth_sec = df_targets['depth_uni'].iloc[nb]
elif df_targets['signal_type'].iloc[nb]== "EBpri cont, EBsec cont" or "EBpri&secP/2 cont":
    depth = df_conts['depth_prim_ldc'].iloc[nc]
    depth_sec = df_conts['depth_sec_uni'].iloc[nc]
elif df_targets['signal_type'].iloc[nb]=="EBpri, EBsec" or "EBpri&secP/2" or "EBpri":
    depth = df_targets['depth_prim_ldc'].iloc[nb]
    depth_sec = df_targets['depth_sec_uni'].iloc[nb]

    
#print(depth,depth_sec)
type_s = df_targets['signal_type'].iloc[nb]
print(type_s)
quart=[]
#Quarter, group, camera
quart.append(df_targets['quarter_event'].iloc[nb])
quart.append(get_group_names(file_simus))
quart.append(df_targets['ncams'].iloc[nb])

stats = {
        'qgc': quart,
        'nb_cont': ncont,
        'mag_star': mag_prim,
        'mag_sec': mag_sec,
        'Ang_dist': dist,
        'depth_prim': depth,
        'depth_sec': depth_sec,
        'Signal_type': type_s
    }
    
param=pd.DataFrame([stats])
print(param)
#Produit

df = pd.read_hdf(file_simus, key='products/aperture/contaminants/metrics_table', start=0, stop=100)
d=pd.DataFrame(data=df,columns=['camera_id','ccd_id','n_mask_efficiency','e_mask_efficiency','s_mask_efficiency','metric_priority','quarter','group','camera'])
print(d)
product=get_quarter_sky(file_simus)
print(product)
print(f"sim{file_name}")

# On regroupe tout proprement
data_to_save = {
    "parametres": param,    # Votre DataFrame 1
    "donnees": d,           # Votre DataFrame 2
    "produit": product      # Votre liste
}

"""with open(f"product_param_{file_name}.json", "w") as f:
    json.dump(data_to_save, f, default=convert_pour_json, indent=4)"""

Pas de contaminant!


NameError: name 'nc' is not defined

In [ ]:
if df_targets['ncon'].iloc[nb]!=0:
    if df_targets['cont_signal'].iloc[nb]==False:
        with h5py.File(file_simus, 'r') as f:
            data = f['contaminants/properties']
            mag_sec = data['magnitude'][:]
            idx_max = np.argmin(mag_sec)
            ra_values = data['ra_bcrs'][idx_max]
            dec_values = data['dec_bcrs'][idx_max]
            dist = ang_dist(dec_values,df_targets['dec'].iloc[nb],ra_values,df_targets['ra'].iloc[nb])
            print('reussi',mag_sec,dist)

In [13]:
product=get_quarter_sky(file_simus)
print(product)

['Q1_nominal_g1', 'Q1_nominal_g2', 'Q1_nominal_g4', 'Q1_extended_g1', 'Q1_extended_g2', 'Q1_extended_g4', 'Q2_nominal_g1', 'Q2_nominal_g3', 'Q2_nominal_g4', 'Q2_extended_g1', 'Q2_extended_g3', 'Q2_extended_g4', 'Q3_nominal_g2', 'Q3_nominal_g3', 'Q3_nominal_g4', 'Q3_extended_g2', 'Q3_extended_g3', 'Q3_extended_g4', 'Q4_nominal_g1', 'Q4_nominal_g2', 'Q4_nominal_g3', 'Q4_extended_g1', 'Q4_extended_g2', 'Q4_extended_g3']


In [18]:
nb=61 #numéro de la simulation 
if nb<10:
    simu=f"0000{nb}"
elif nb<100:
    simu=f"000{nb}"
elif nb>=100:
    simu=f"00{nb}"
file_name=f"sim{simu}"
file_simus = f"{path_local}/{file_name}.hdf5"
file_feather_all = '/net/GSP/nas12c/plato/FichiersParams/sim00000_00420_AllParameters.ftr'
df_targets = pd.read_feather(file_feather_all)
if df_targets['ncon'].iloc[nb]!=0:
    if df_targets['cont_signal'].iloc[nb]==False:
        with h5py.File(file_simus, 'r') as f:
            data = f['contaminants/properties']
            mag_sec = data['magnitude'][:]
            idx_max = np.argmin(mag_sec)
            ra_values = data['ra_bcrs'][idx_max]
            dec_values = data['dec_bcrs'][idx_max]
            dist = ang_dist(dec_values,df_targets['dec'].iloc[nb],ra_values,df_targets['ra'].iloc[nb])
            print('reussi',mag_sec,dist)

reussi [16.9716 16.491  15.7327 17.4444 17.2093] 42.808429412499


In [24]:
print(type(param))

<class 'pandas.core.frame.DataFrame'>


In [13]:
def convert_pour_json(obj):
    # 1. Gestion des DataFrames Pandas
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    
    # 2. Gestion des Series Pandas (si besoin)
    if isinstance(obj, pd.Series):
        return obj.tolist()

    # 3. Gestion des types Numpy (Tableaux et Scalaires)
    if isinstance(obj, (np.ndarray, np.generic)):
        return obj.tolist()
    
    # 4. Gestion spécifique des flottants (si tolist() ne suffit pas)
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)

    raise TypeError(f"L'objet de type {type(obj)} n'est toujours pas sérialisable")